# 02 · Congelación del baseline

Este notebook toma el **retrato completo del sistema baseline** — el agente tal y como quedó en la entrega 2 (búsqueda densa, sin middleware, prompt congelado) — y lo deja fijado antes de tocar una sola mejora. Es media tabla del informe y **no se puede reconstruir a posteriori**: cualquier mejora que se cuele antes de esta tirada contamina la comparación para siempre.

Tres tandas, cada una con su papel:

| Tanda | Fichero | Papel |
|---|---|---|
| Propio (20) | `golden/golden_set.jsonl` | **La tabla del informe** (enunciado §5: baseline vs final sobre el golden propio) |
| Oficial (20) | `golden/oficial_20.jsonl` | Diagnóstico y comparabilidad con el material del curso |
| Huecos (5) | `golden/huecos_humo.jsonl` | Ensayo del comportamiento `fuente='ninguna'` (≥2 de las 10 ciegas serán así) |

Coste estimado de la tirada completa: **30-60 céntimos** (45 preguntas a ~0,4-1,2 ¢). Duración: 10-15 minutos, con progreso por pregunta. `temperature=0` en todo: la tirada es reproducible.

**Regla a partir de la última celda:** el baseline queda etiquetado en git (`git tag baseline`) y estos CSV no se regeneran. Toda mejora se mide contra estas cifras.

In [ ]:
import json
import sys
from pathlib import Path

RAIZ = Path.cwd() if (Path.cwd() / "agente").exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ))

import os
import pandas as pd

if not os.environ.get("OPENROUTER_API_KEY"):
    from getpass import getpass
    os.environ["OPENROUTER_API_KEY"] = getpass("OPENROUTER_API_KEY: ")

from agente import evaluadores, interfaz, retrieval

# El baseline se define por su punto de intercambio: densa. Si esto falla,
# alguien descongeló antes de tiempo.
assert retrieval.buscar_agente is retrieval.buscar_densa, \
    "buscar_agente ya no es la densa: esto NO es el baseline"

propio = evaluadores.cargar_golden(RAIZ / "golden/golden_set.jsonl")
oficial = evaluadores.cargar_golden(RAIZ / "golden/oficial_20.jsonl")
huecos = evaluadores.cargar_golden(RAIZ / "golden/huecos_humo.jsonl")
fams = pd.Series([g["familia"] for g in propio]).value_counts().to_dict()
assert len(propio) == 20 and fams.get("comparativa", 0) >= 6, fams
print(f"propio: {len(propio)} {fams} · oficial: {len(oficial)} · "
      f"huecos: {len(huecos)}")
print("Baseline verificado: search_filings = densa + filtros, sin middleware.")

## Tanda 1 — Golden propio (la tabla del informe)

Cada pregunta corre en su propio `thread_id`; la columna `recall` se mide con el retrieval **del baseline** (densa+filtros sobre la consulta reescrita, que es el régimen en el que el agente consulta). Un error en una pregunta produce una fila con `error`, no un crash.

In [ ]:
df_propio = interfaz.evaluar(str(RAIZ / "golden/golden_set.jsonl"),
                             etiqueta="baseline_propio")

## Tanda 2 — Golden oficial (diagnóstico)

Las mismas 20 con las que el profesor razona en clase. No es la tabla del informe, pero sí el mejor contraste: aquí sabemos qué esperaba el curso de cada pregunta (la trampa de Alphabet en of-012, los huecos de capex en of-018, la guidance de of-017…).

In [ ]:
df_oficial = interfaz.evaluar(str(RAIZ / "golden/oficial_20.jsonl"),
                              etiqueta="baseline_oficial")

## Tanda 3 — Huecos (el ensayo de las ciegas)

Cinco preguntas **sin respuesta en el corpus**. Aquí la columna que importa es `cifra_ok` con nuestra mejora de evaluador: `True` solo si el agente respondió sin cifra y con `fuente='ninguna'`; una cifra inventada puntúa `False` (el evaluador del taller devolvía `None` y la invención salía gratis). "Que el agente conteste fuente='ninguna' en vez de inventarse una cifra es la mitad del examen".

In [ ]:
df_huecos = interfaz.evaluar(str(RAIZ / "golden/huecos_humo.jsonl"),
                             etiqueta="baseline_huecos")

## Consolidado y coste real de la tirada

In [ ]:
resumen = pd.DataFrame([
    evaluadores.resumir(df_propio, "baseline · propio (20)"),
    evaluadores.resumir(df_oficial, "baseline · oficial (20)"),
    evaluadores.resumir(df_huecos, "baseline · huecos (5)"),
]).set_index("sistema").round(3)
display(resumen)

(RAIZ / "resultados").mkdir(exist_ok=True)
resumen.to_csv(RAIZ / "resultados/baseline_resumen.csv")

coste_total = sum(df["coste_usd"].dropna().sum()
                  for df in (df_propio, df_oficial, df_huecos))
print(f"Coste real de la tirada: {coste_total*100:.1f} ¢")
print("Guardado: resultados/baseline_resumen.csv (+ los tres eval_*.csv)")

print("\nDesglose por familia — propio:")
display(evaluadores.por_familia(df_propio).round(2))
print("Fallos por tanda:")
for nombre, df in [("propio", df_propio), ("oficial", df_oficial),
                   ("huecos", df_huecos)]:
    malas = df[(df[["cita_ok", "cifra_ok", "tool_ok"]] == False).any(axis=1)
               | df["error"].notna()]["id"].tolist()
    print(f"  {nombre}: {malas if malas else 'ninguno'}")

## Inspector de fallos

Para entender un fallo no basta la fila del CSV: hay que **ver la trayectoria**. La celda siguiente define `inspeccionar(id)`: reejecuta esa pregunta en un hilo nuevo con `pretty_trace` (cuesta lo que una pregunta, ~0,5-1 ¢). Clasifica cada fallo en su causa — retrieval (no encontró), enrutado (herramienta equivocada), cita (no ancló) o cifra (número mal) — porque **la causa decide qué mejora lo ataca**: el flip a híbrida ataca retrieval, el middleware ataca cifra, el prompt ataca enrutado. Ese mapa fallo→causa→mejora es la sección central del informe.

In [ ]:
TODAS = {g["id"]: g for g in propio + oficial + huecos}

def inspeccionar(ident: str):
    from agente import trazas
    g = TODAS[ident]
    print(f"[{ident}] {g['familia']} · esperada: "
          f"{g.get('cifra_esperada')} · tools: {g['herramienta_esperada']}")
    print(f"P: {g['pregunta']}\n")
    r = interfaz.responder(g["pregunta"], thread_id=f"inspeccion-{ident}")
    trazas.pretty_trace(r)
    print(f"\n[{r['latencia_s']:.1f} s · {r['coste_usd']*100:.2f} ¢]")
    return r

# Ejemplo (descomenta el fallo que quieras mirar):
# _ = inspeccionar("of-012")

## Congelación

Con las tres tandas arriba y los CSV escritos, el baseline se congela **fuera** de este notebook, en el terminal:

```bash
git add .
git commit -m "Baseline congelado: tirada baseline (propio/oficial/huecos) sobre el golden propio final"
git tag -fa baseline -m "Sistema baseline: densa+filtros, sin middleware, prompt v1"
git push && git push --force origin baseline
```

`-f` y `--force` solo son necesarios si el tag `baseline` ya existía de una tirada anterior: mueven el tag al commit nuevo.

A partir del tag: (1) `buscar_agente` pasa a la configuración ganadora del notebook 01, (2) entra el middleware (límites + verificación por concepto), (3) cada mejora se re-mide con `evaluar()` y se compara contra `resultados/eval_baseline_propio.csv`. Nada de esta página se reejecuta después del tag.

In [ ]:
print("NOTEBOOK 02 COMPLETADO — baseline retratado.")
print("Siguiente paso en el terminal: commit + git tag baseline + push --tags")
print("Ficheros para compartir: resultados/eval_baseline_propio.csv, "
      "eval_baseline_oficial.csv, eval_baseline_huecos.csv, "
      "baseline_resumen.csv")